# 04. Multi-Task Triple-Head Deep Learning GRU (h=10) — 200 Users 5G Dataset (2D Localization)

This notebook trains a **Multi-Task Triple-Head Gated Recurrent Unit (GRU)** model with **History Depth $h=10$** ($L=11$ sequence window timesteps) on the 5G NR 3.0 GHz **200-User Dataset** (`99,589 samples`).

### Target Representation: 2D Relative Horizontal Coordinates $(\Delta x, \Delta y)$
- **2D Horizontal Targets:** $(\Delta x, \Delta y) = (x_{\text{UE}} - x_{\text{BS}}, y_{\text{UE}} - y_{\text{BS}})$.
- **Device Context:** `ue_height` is provided as input static hardware context (essential for AoA elevation geometry $\phi_{\text{el}}$), while the network predicts the unknown 2D horizontal location $(\Delta x, \Delta y)$.
- **Target Standardization:** Target coordinates $(\Delta x, \Delta y)$ are standardized $(\mu=0, \sigma=1)$ during training for smooth, stable loss curves.
- **Strict Trajectory Alignment:** Uses 1-to-1 array masking (`uids == uid`) to guarantee 100% exact alignment between ground truth trajectory waypoints and predictions.

In [ ]:
import sys, os, time, warnings, json, math, datetime, glob, copy
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.linalg import inv
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))

from pipelines.multi_user_200_pipeline import load_200_users, make_unseen_user_split
from pipelines.multi_user_pipeline_regression import _read_bs_position_3d

RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
OUT_DIR = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'multi_user_poc' / f'gru_h10_{RUN_TIMESTAMP}'
PLOT_DIR = OUT_DIR / 'trajectory_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUT_DIR}')
print(f'Trajectory plots directory: {PLOT_DIR}')

## Global Hyperparameters & Configurable Parameters (TOP-LEVEL)

In [ ]:
# ── Top-Level Configurable Parameters ───────────────────────────────────
SEED                = 42         # Random seed for reproducibility
TRAIN_USER_RATIO    = 0.80       # 80% Train Users (160) / 20% Unseen Test Users (40)
H_TARGET            = 10         # History depth (h=10 sequence window L=11)
SEQUENCE_LENGTH     = H_TARGET + 1 # L = 11 timesteps

# Training & Architecture Parameters
BATCH_SIZE          = 512        # Reduced batch size (512) for rich stochastic updates
EPOCHS              = 35         # Optimized Epoch Budget (35) with Cosine Annealing
LEARNING_RATE       = 5e-4       # Default initial learning rate
WEIGHT_DECAY        = 1e-3       # Increased L2 Weight decay regularization
GRU_HIDDEN_DIM      = 128        # GRU hidden state dimension
GRU_NUM_LAYERS      = 2          # Number of stacked GRU layers
DROPOUT_RATE        = 0.30       # Dropout (0.30) for regularization

# Multi-Task Head Loss Weights
LAMBDA_POS          = 1.0        # Head 1: 2D Position Loss Weight
LAMBDA_SPEED        = 0.2        # Head 2: Physical Speed Loss Weight
LAMBDA_UNCERTAINTY  = 0.05       # Head 3: Normalized Uncertainty Loss Weight

# LR Finder Parameters
LR_START            = 1e-7       # Lower starting LR for smooth baseline
LR_END              = 1e-1       # Ending LR for sweep
LR_STEPS            = 150        # Number of mini-batches for LR sweep

# Kinematic Post-Processing Parameters
PROCESS_NOISE_STD   = 0.5        # Kalman process noise std
R_STD               = 15.0       # Kalman measurement noise std

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')
print(f'Configured Multi-Task GRU (2D Target): h={H_TARGET} | Epochs={EPOCHS} | Batch Size={BATCH_SIZE}')

## Load 200-User Dataset & Apply 4° AoA Noise Model

In [ ]:
data_base = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25'
dirs = sorted(list(data_base.glob('sim_data_200users_*')))
if not dirs:
    raise FileNotFoundError('No sim_data_200users_* dataset directories found!')
DATA_DIR = dirs[-1]
print(f'Loading 200-User dataset from: {DATA_DIR}')

df_raw = load_200_users(DATA_DIR)
print(f'Loaded {len(df_raw):,} records across {len(df_raw["user_id"].unique())} diverse mobile users.')

# Compute 2D Relative Displacements (target_x, target_y) Relative to Serving BS
bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
ue_xy = np.column_stack([df_raw['x_pos'], df_raw['y_pos']])
delta  = ue_xy - bs_pos[:2]
df_raw['target_x'], df_raw['target_y'] = delta[:,0], delta[:,1]
print(f'Serving Base Station 2D Location: [{bs_pos[0]:.1f}, {bs_pos[1]:.1f}] m')
print('Targets computed as 2D relative displacement (UE_xy - BS_xy).')

AOA_NOISE_STD_DEG = 4.0
AOA_QUANT_STEP_DEG = 5.0
mask_ant = df_raw['n_antennas'] > 1
rng_noise = np.random.RandomState(SEED)
df_raw.loc[mask_ant, 'aoa_azimuth'] = df_raw.loc[mask_ant, 'aoa_azimuth'].values + (AOA_NOISE_STD_DEG * rng_noise.randn(mask_ant.sum())).astype(np.float32)
df_raw.loc[mask_ant, 'aoa_elevation'] = df_raw.loc[mask_ant, 'aoa_elevation'].values + (AOA_NOISE_STD_DEG * rng_noise.randn(mask_ant.sum())).astype(np.float32)
df_raw.loc[mask_ant, 'aoa_azimuth'] = np.round(df_raw.loc[mask_ant, 'aoa_azimuth'] / AOA_QUANT_STEP_DEG) * AOA_QUANT_STEP_DEG
df_raw.loc[mask_ant, 'aoa_elevation'] = np.round(df_raw.loc[mask_ant, 'aoa_elevation'] / AOA_QUANT_STEP_DEG) * AOA_QUANT_STEP_DEG
df_raw.loc[~mask_ant, ['aoa_azimuth', 'aoa_elevation']] = 0.0
print('Noise model applied.')

## Construct Multi-Task Sequence Dataset (2D Target: Position, Speed, Uncertainty)

In [ ]:
class MultiTaskCSIDataset(Dataset):
    def __init__(self, df, h=10, sig_mean=None, sig_std=None, stat_mean=None, stat_std=None, targ_mean=None, targ_std=None, speed_mean=None, speed_std=None):
        self.samples = []
        signal_cols = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation', 'delta_t']
        static_cols = ['n_antennas', 'antenna_gain_db', 'ue_height']
        target_cols = ['target_x', 'target_y']  # 2D Horizontal Target
        
        df = df.copy()
        self.sig_mean  = sig_mean  if sig_mean  is not None else df[signal_cols].mean().values
        self.sig_std   = sig_std   if sig_std   is not None else df[signal_cols].std().values + 1e-6
        self.stat_mean = stat_mean if stat_mean is not None else df[static_cols].mean().values
        self.stat_std  = stat_std  if stat_std  is not None else df[static_cols].std().values + 1e-6
        self.targ_mean = targ_mean if targ_mean is not None else df[target_cols].mean().values
        self.targ_std  = targ_std  if targ_std  is not None else df[target_cols].std().values + 1e-6
        self.speed_mean= speed_mean if speed_mean is not None else float(df['speed_m_s'].mean())
        self.speed_std = speed_std  if speed_std  is not None else float(df['speed_m_s'].std()) + 1e-6
        
        df[signal_cols] = (df[signal_cols] - self.sig_mean) / self.sig_std
        df[static_cols] = (df[static_cols] - self.stat_mean) / self.stat_std
        df[target_cols] = (df[target_cols] - self.targ_mean) / self.targ_std
        df['norm_speed'] = (df['speed_m_s'] - self.speed_mean) / self.speed_std
        
        L = h + 1
        for uid, udf in df.groupby('user_id'):
            udf = udf.sort_values('step_index').reset_index(drop=True)
            n_steps = len(udf)
            if n_steps < L:
                continue
            
            sigs   = udf[signal_cols].values.astype(np.float32)
            stats  = udf[static_cols].values.astype(np.float32)
            targs  = udf[target_cols].values.astype(np.float32)
            speeds = udf['norm_speed'].values.astype(np.float32)
            u_ids  = udf['user_id'].values
            dts    = udf['delta_t'].values
            
            for i in range(h, n_steps):
                seq_sig   = sigs[i-h:i+1]
                stat_vec  = stats[i]
                targ_vec  = targs[i]
                speed_val = np.array([speeds[i]], dtype=np.float32)
                self.samples.append({
                    'seq': seq_sig,
                    'static': stat_vec,
                    'target': targ_vec,
                    'speed': speed_val,
                    'user_id': u_ids[i],
                    'delta_t': dts[i]
                })
                
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'seq': torch.tensor(s['seq'], dtype=torch.float32),
            'static': torch.tensor(s['static'], dtype=torch.float32),
            'target': torch.tensor(s['target'], dtype=torch.float32),
            'speed': torch.tensor(s['speed'], dtype=torch.float32),
            'user_id': s['user_id'],
            'delta_t': s['delta_t']
        }

df_split, train_users, test_users = make_unseen_user_split(df_raw, train_ratio=TRAIN_USER_RATIO, seed=SEED)
df_train = df_split[df_split['split']=='train'].reset_index(drop=True)
df_test  = df_split[df_split['split']=='test'].reset_index(drop=True)

train_ds = MultiTaskCSIDataset(df_train, h=H_TARGET)
test_ds  = MultiTaskCSIDataset(df_test, h=H_TARGET,
                               sig_mean=train_ds.sig_mean, sig_std=train_ds.sig_std,
                               stat_mean=train_ds.stat_mean, stat_std=train_ds.stat_std,
                               targ_mean=train_ds.targ_mean, targ_std=train_ds.targ_std,
                               speed_mean=train_ds.speed_mean, speed_std=train_ds.speed_std)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Multi-Task Dataset Built (2D Target): {len(train_ds):,} Train Sequences ({len(train_users)} Users) | {len(test_ds):,} Test Sequences ({len(test_users)} Users)')
print(f'Batch Size: {BATCH_SIZE} ({len(train_loader)} batches per epoch)')

## Define Multi-Task Triple-Head GRU Network (2D Output)

In [ ]:
class MultiTaskGRU2DNet(nn.Module):
    def __init__(self, input_dim=5, static_dim=3, hidden_dim=128, num_layers=2, dropout=0.30):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        
        fusion_dim = hidden_dim + static_dim
        self.shared_trunk = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LeakyReLU(0.1),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout)
        )
        
        # Head 1: 2D Position Head (2D Coordinates)
        self.head_pos = nn.Sequential(
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 2)
        )
        
        # Head 2: Physical Speed Head
        self.head_speed = nn.Sequential(
            nn.Linear(256, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1)
        )
        
        # Head 3: 90% Uncertainty Radius Head
        self.head_unc = nn.Sequential(
            nn.Linear(256, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1),
            nn.Softplus()
        )
        
    def forward(self, x_seq, x_static):
        gru_out, _ = self.gru(x_seq)
        h_last = gru_out[:, -1, :]
        fused = torch.cat([h_last, x_static], dim=1)
        feat = self.shared_trunk(fused)
        
        pred_pos   = self.head_pos(feat)
        pred_speed = self.head_speed(feat)
        pred_unc   = self.head_unc(feat)
        return pred_pos, pred_speed, pred_unc

model = MultiTaskGRU2DNet(input_dim=5, static_dim=3, hidden_dim=GRU_HIDDEN_DIM, num_layers=GRU_NUM_LAYERS, dropout=DROPOUT_RATE).to(device)
print(model)

## Learning Rate Finder Sweep ($10^{-7} \to 10^{-1}$)

In [ ]:
# ── PyTorch Learning Rate Finder (1e-7 -> 1e-1) ──────────────────────────
def find_learning_rate(model, train_loader, lr_start=1e-7, lr_end=1e-1, num_steps=150, beta=0.98):
    init_state = copy.deepcopy(model.state_dict())
    model.train()
    
    mult = (lr_end / lr_start) ** (1.0 / num_steps)
    lr = lr_start
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion_pos   = nn.SmoothL1Loss()
    criterion_speed = nn.SmoothL1Loss()
    
    avg_loss = 0.0
    best_loss = float('inf')
    batch_num = 0
    lrs, losses = [], []
    
    print(f'Starting Extended LR Finder Sweep from {lr_start:.1e} to {lr_end:.1e} over {num_steps} steps...')
    
    for batch in train_loader:
        batch_num += 1
        seq   = batch['seq'].to(device)
        stat  = batch['static'].to(device)
        t_pos = batch['target'].to(device)
        t_spd = batch['speed'].to(device)
        
        optimizer.zero_grad()
        p_pos, p_spd, p_unc = model(seq, stat)
        
        loss1 = criterion_pos(p_pos, t_pos)
        loss2 = criterion_speed(p_spd, t_spd)
        loss = LAMBDA_POS * loss1 + LAMBDA_SPEED * loss2
        
        avg_loss = beta * avg_loss + (1 - beta) * loss.item()
        smoothed_loss = avg_loss / (1 - beta ** batch_num)
        
        if batch_num > 1 and smoothed_loss > 4.0 * best_loss:
            break
        if smoothed_loss < best_loss or batch_num == 1:
            best_loss = smoothed_loss
            
        lrs.append(lr)
        losses.append(smoothed_loss)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        lr *= mult
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr
            
        if batch_num >= num_steps: break
            
    model.load_state_dict(init_state)
    losses_arr = np.array(losses)
    grad = np.gradient(losses_arr)
    best_idx = np.argmin(grad)
    suggested_lr = lrs[best_idx]
    
    plt.figure(figsize=(9, 5))
    plt.plot(lrs, losses, 'b-', lw=2.5, label='Smoothed Multi-Task Loss')
    plt.axvline(suggested_lr, color='crimson', linestyle='--', lw=2, label=f'Suggested LR: {suggested_lr:.2e}')
    plt.xscale('log')
    plt.xlabel('Learning Rate (Log Scale)', fontsize=11)
    plt.ylabel('Normalized Loss', fontsize=11)
    plt.title('Multi-Task Learning Rate Finder Sweep ($10^{-7} \to 10^{-1}$)', fontsize=12, fontweight='bold')
    plt.grid(True, which='both', alpha=0.3)
    plt.legend(fontsize=10)
    plt.savefig(OUT_DIR / 'lr_finder_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'LR Finder Complete! Suggested Learning Rate: {suggested_lr:.2e}')
    return suggested_lr

suggested_lr = find_learning_rate(model, train_loader, lr_start=LR_START, lr_end=LR_END, num_steps=LR_STEPS)
if suggested_lr > 1e-4:
    LEARNING_RATE = suggested_lr
else:
    LEARNING_RATE = 5e-4  # Default robust LR floor
print(f'Active Learning Rate set to: {LEARNING_RATE:.2e}')

## Train Multi-Task Triple-Head GRU Network (2D MAE Evaluation)

In [ ]:
criterion_pos   = nn.SmoothL1Loss()
criterion_speed = nn.SmoothL1Loss()
criterion_unc   = nn.SmoothL1Loss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_mae = float('inf')
best_model_state = None
best_epoch = 0

print(f'Starting Multi-Task GRU Training for {EPOCHS} Epochs with LR={LEARNING_RATE:.2e} on {device}...')
print('='*110)
t_total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    t_ep_start = time.time()
    model.train()
    tr_loss = 0.0
    tr_errs = []
    
    for batch in train_loader:
        seq   = batch['seq'].to(device)
        stat  = batch['static'].to(device)
        t_pos = batch['target'].to(device)
        t_spd = batch['speed'].to(device)
        
        optimizer.zero_grad()
        p_pos, p_spd, p_unc = model(seq, stat)
        
        norm_errs = torch.norm(p_pos.detach() - t_pos, dim=1, keepdim=True)
        
        loss1 = criterion_pos(p_pos, t_pos)
        loss2 = criterion_speed(p_spd, t_spd)
        loss3 = criterion_unc(p_unc, norm_errs)
        
        loss = LAMBDA_POS * loss1 + LAMBDA_SPEED * loss2 + LAMBDA_UNCERTAINTY * loss3
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tr_loss += loss.item() * len(t_pos)
        
        p_pos_m = p_pos.detach().cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        t_pos_m = t_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        tr_errs.extend(np.linalg.norm(p_pos_m - t_pos_m, axis=1))
        
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()
    tr_loss /= len(train_ds)
    tr_mae = np.mean(tr_errs)
    t_ep_time = time.time() - t_ep_start
    
    model.eval()
    val_loss = 0.0
    val_errs = []
    with torch.no_grad():
        for batch in test_loader:
            seq   = batch['seq'].to(device)
            stat  = batch['static'].to(device)
            t_pos = batch['target'].to(device)
            t_spd = batch['speed'].to(device)
            p_pos, p_spd, p_unc = model(seq, stat)
            
            norm_errs = torch.norm(p_pos - t_pos, dim=1, keepdim=True)
            loss1 = criterion_pos(p_pos, t_pos)
            loss2 = criterion_speed(p_spd, t_spd)
            loss3 = criterion_unc(p_unc, norm_errs)
            loss  = LAMBDA_POS * loss1 + LAMBDA_SPEED * loss2 + LAMBDA_UNCERTAINTY * loss3
            val_loss += loss.item() * len(t_pos)
            
            pred_m = p_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
            targ_m = t_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
            val_errs.extend(np.linalg.norm(pred_m - targ_m, axis=1))
            
    val_loss /= len(test_ds)
    val_mae = np.mean(val_errs)
    
    is_best = ''
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, OUT_DIR / 'best_gru_model.pt')
        is_best = ' [BEST *]'
        
    t_elapsed = time.time() - t_total_start
    eta_min = (t_elapsed / epoch) * (EPOCHS - epoch) / 60.0
    
    if epoch % 2 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f'Epoch {epoch:2d}/{EPOCHS:2d} | Train Loss: {tr_loss:.4f} | Train 2D MAE: {tr_mae:6.3f}m | Val Loss: {val_loss:.4f} | Val 2D MAE: {val_mae:6.3f}m | LR: {current_lr:.2e} | Ep Time: {t_ep_time:4.1f}s | ETA: {eta_min:4.1f}min{is_best}')

t_total = time.time() - t_total_start
print('='*110)
print(f'Restoring Best GRU Model Weights from Epoch {best_epoch} (Best Val 2D MAE: {best_val_mae:.3f}m)...')
model.load_state_dict(best_model_state)
print(f'Multi-Task GRU Training Completed in {t_total:.2f} seconds ({t_total/60:.2f} minutes)!')

## Evaluate Best Multi-Task GRU Model on Unseen Test Users (2D Performance)

In [ ]:
model.eval()
preds_pos_list, preds_speed_list, preds_unc_list = [], [], []
targs_pos_list, targs_speed_list = [], []
uids_list, dts_list = [], []

with torch.no_grad():
    for batch in test_loader:
        seq   = batch['seq'].to(device)
        stat  = batch['static'].to(device)
        t_pos = batch['target'].to(device)
        t_spd = batch['speed'].to(device)
        p_pos, p_spd, p_unc = model(seq, stat)
        
        pred_pos_m   = p_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        targ_pos_m   = t_pos.cpu().numpy() * train_ds.targ_std + train_ds.targ_mean
        pred_speed_m = p_spd.cpu().numpy() * train_ds.speed_std + train_ds.speed_mean
        targ_speed_m = t_spd.cpu().numpy() * train_ds.speed_std + train_ds.speed_mean
        pred_unc_m   = p_unc.cpu().numpy() * np.mean(train_ds.targ_std)
        
        preds_pos_list.append(pred_pos_m)
        targs_pos_list.append(targ_pos_m)
        preds_speed_list.append(pred_speed_m)
        targs_speed_list.append(targ_speed_m)
        preds_unc_list.append(pred_unc_m)
        uids_list.extend(batch['user_id'].numpy())
        dts_list.extend(batch['delta_t'].numpy())

preds_raw   = np.vstack(preds_pos_list)
targs       = np.vstack(targs_pos_list)
preds_speed = np.vstack(preds_speed_list)
targs_speed = np.vstack(targs_speed_list)
preds_unc   = np.vstack(preds_unc_list)
uids        = np.array(uids_list)
dts         = np.array(dts_list)

errs_2d  = np.linalg.norm(preds_raw - targs, axis=1)
mae_2d   = np.mean(errs_2d)
mae_x    = mean_absolute_error(targs[:, 0], preds_raw[:, 0])
mae_y    = mean_absolute_error(targs[:, 1], preds_raw[:, 1])
mae_spd  = mean_absolute_error(targs_speed, preds_speed)
p50      = np.percentile(errs_2d, 50)
p90      = np.percentile(errs_2d, 90)

print(f'=== BEST MULTI-TASK GRU 2D BENCHMARK (RESTORED FROM EPOCH {best_epoch}) ===')
print(f'Head 1: 2D Horizontal Position MAE:  {mae_2d:.3f} meters')
print(f'        X-Axis MAE:                   {mae_x:.3f} meters')
print(f'        Y-Axis MAE:                   {mae_y:.3f} meters')
print(f'        50th Percentile (Median):     {p50:.3f} meters')
print(f'        90th Percentile Error:       {p90:.3f} meters')
print(f'Head 2: Physical Speed MAE:           {mae_spd:.3f} m/s')
print(f'Head 3: Mean Predicted R90:          {np.mean(preds_unc):.3f} meters')

## Kinematic Post-Processing: 2D Forward KF & RTS Smoother

In [ ]:
def run_kalman_and_rts_2d(y_true, y_pred, delta_t_vec, process_noise_std=0.5, R_std=15.0):
    N = len(y_pred)
    if N == 0: return y_pred, y_pred
    H = np.zeros((2, 4)); H[0, 0], H[1, 1] = 1.0, 1.0
    R = (R_std ** 2) * np.eye(2)
    x_pred = np.zeros((N, 4)); P_pred = np.zeros((N, 4, 4))
    x_filt = np.zeros((N, 4)); P_filt = np.zeros((N, 4, 4))
    x_filt[0, :2] = y_pred[0]; P_filt[0] = np.eye(4) * 100.0
    x_pred[0], P_pred[0] = x_filt[0], P_filt[0]
    for t in range(1, N):
        dt = max(0.01, float(delta_t_vec[t]))
        F = np.eye(4); F[0, 2], F[1, 3] = dt, dt
        q = (process_noise_std ** 2)
        Q = np.diag([q*(dt**2), q*(dt**2), q, q])
        x_p = F @ x_filt[t-1]; P_p = F @ P_filt[t-1] @ F.T + Q
        x_pred[t], P_pred[t] = x_p, P_p
        z_t = y_pred[t]; y_k = z_t - H @ x_p
        S_k = H @ P_p @ H.T + R; K_k = P_p @ H.T @ inv(S_k)
        x_filt[t] = x_p + K_k @ y_k
        P_filt[t] = (np.eye(4) - K_k @ H) @ P_p
    x_smooth = np.zeros((N, 4)); P_smooth = np.zeros((N, 4, 4))
    x_smooth[-1], P_smooth[-1] = x_filt[-1], P_filt[-1]
    for t in range(N - 2, -1, -1):
        dt = max(0.01, float(delta_t_vec[t+1]))
        F = np.eye(4); F[0, 2], F[1, 3] = dt, dt
        C_k = P_filt[t] @ F.T @ inv(P_pred[t+1])
        x_smooth[t] = x_filt[t] + C_k @ (x_smooth[t+1] - x_pred[t+1])
        P_smooth[t] = P_filt[t] + C_k @ (P_smooth[t+1] - P_pred[t+1]) @ C_k.T
    return x_filt[:, :2], x_smooth[:, :2]

kf_preds  = np.zeros_like(preds_raw)
rts_preds = np.zeros_like(preds_raw)

for uid in test_users:
    u_mask = (uids == uid)
    if not np.any(u_mask): continue
    y_u_true = targs[u_mask]
    y_u_pred = preds_raw[u_mask]
    dt_u     = dts[u_mask]
    x_kf, x_rts = run_kalman_and_rts_2d(y_u_true, y_u_pred, dt_u, process_noise_std=PROCESS_NOISE_STD, R_std=R_STD)
    kf_preds[u_mask]  = x_kf
    rts_preds[u_mask] = x_rts

mae_kf  = np.mean(np.linalg.norm(kf_preds - targs, axis=1))
mae_rts = np.mean(np.linalg.norm(rts_preds - targs, axis=1))
red_rts = ((mae_2d - mae_rts) / mae_2d) * 100.0

print(f'=== BEST MULTI-TASK GRU + KINEMATIC SMOOTHING 2D BENCHMARK (h=10, EPOCH {best_epoch}) ===')
print(f'Raw Multi-Task GRU 2D MAE: {mae_2d:.3f} meters')
print(f'Forward Linear KF:         {mae_kf:.3f} meters')
print(f'RTS Kalman Smoother:       {mae_rts:.3f} meters ({red_rts:+5.1f}%)')

## Trajectory Tracking Plots Across 5 Diverse Unseen Test Users

In [ ]:
sorted_test_uids = sorted(list(test_users))
sample_5_uids = sorted_test_uids[:5]

print(f'Generating and saving trajectory plots for 5 unseen test users: {sample_5_uids}')
print(f'Saving trajectory plots to: {PLOT_DIR}')

for u_idx, uid in enumerate(sample_5_uids, 1):
    u_mask = (uids == uid)
    if not np.any(u_mask): continue
    
    gt_s  = targs[u_mask]
    raw_s = preds_raw[u_mask]
    rts_s = rts_preds[u_mask]
    
    u_mae_raw = np.mean(np.linalg.norm(raw_s - gt_s, axis=1))
    u_mae_rts = np.mean(np.linalg.norm(rts_s - gt_s, axis=1))
    
    plt.figure(figsize=(10, 7.5))
    plt.plot(gt_s[:, 0], gt_s[:, 1], 'k-', lw=3.0, label=f'Ground Truth (User {uid})')
    plt.scatter(raw_s[:, 0], raw_s[:, 1], color='mediumpurple', alpha=0.55, s=22, label=f'Best Multi-Task GRU (MAE: {u_mae_raw:.2f}m)')
    plt.plot(rts_s[:, 0], rts_s[:, 1], 'g-', lw=2.2, label=f'RTS Smoother (MAE: {u_mae_rts:.2f}m)')
    plt.xlabel('X Relative to Serving BS (meters)', fontsize=11)
    plt.ylabel('Y Relative to Serving BS (meters)', fontsize=11)
    plt.title(f'User {uid} Trajectory Tracking (Best GRU Epoch {best_epoch} + RTS Smoother)', fontsize=12, fontweight='bold')
    plt.legend(fontsize=10, loc='upper right')
    plt.grid(True, alpha=0.3)
    
    plot_path = PLOT_DIR / f'trajectory_user_{uid}.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  [{u_idx}/5] Saved: {plot_path.name} | Raw MAE: {u_mae_raw:.2f}m | RTS MAE: {u_mae_rts:.2f}m')